1. “Suggest a Christopher Nolan Movie”
intent: recommend
type: movie
director: Christopher Nolan
2. “I want to watch a comedy movie while having my lunch”
intent: recommend
type: movie
genre: comedy
extra nuance: “while having my lunch” → maybe treat as “something light / not too long”
could map to: duration < some threshold, or just ignore at first.
3. “Suggest a good horror movie”
intent: recommend
type: movie (implicit)
genre: horror
“good” → maybe prefer higher‑rated / popular ones, or just pick a strong candidate.
4. “Suggest a TV show to watch. I am in the mood for some thriller”
intent: recommend
type: TV Show
genre: thriller
5. “Tell about a famous Indian Movie that I can watch”
intent: recommend + describe
type: movie
country: India
“famous” → prefer popular/well‑known (you might approximate via recency or presence in description, but at the start you can just pick a random one).
Response should include a short description, not just the title.
6. “Suggest a Robert Downey Jr movie”
intent: recommend
type: movie
actor in cast: Robert Downey Jr
7. “I want to see some classic American movie”
intent: recommend
type: movie
country: United States (or contains “United States” / “USA”)
“classic” → translate to an older release year range, e.g. before 2000, or before 1990.
8. “Suggest a comedy tv show”
intent: recommend
type: TV Show
genre: comedy


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn

In [3]:
df = pd.read_csv("/Users/jaimeet/Documents/Movie Recommender/data/netflix_titles.csv")
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [5]:
for c in df.columns:
    if df[c].isnull().any():
        df[c] = df[c].fillna('unknown')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      8807 non-null   object
 4   cast          8807 non-null   object
 5   country       8807 non-null   object
 6   date_added    8807 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8807 non-null   object
 9   duration      8807 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [7]:
df.shape

(8807, 12)

Columns I need: title,director,cast,country,release_year,rating,duration,description.

In [8]:
df['overview'] = df['title'] + " " + df['listed_in'] + " " + df['description'] + " " + df['cast']

In [9]:
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,unknown,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",Dick Johnson Is Dead Documentaries As her fath...
1,s2,TV Show,Blood & Water,unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...","Blood & Water International TV Shows, TV Drama..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",unknown,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,"Ganglands Crime TV Shows, International TV Sho..."
3,s4,TV Show,Jailbirds New Orleans,unknown,unknown,unknown,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...","Jailbirds New Orleans Docuseries, Reality TV F..."
4,s5,TV Show,Kota Factory,unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,"Kota Factory International TV Shows, Romantic ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a...","Zodiac Cult Movies, Dramas, Thrillers A politi..."
8803,s8804,TV Show,Zombie Dumb,unknown,unknown,unknown,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g...","Zombie Dumb Kids' TV, Korean TV Shows, TV Come..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...,"Zombieland Comedies, Horror Movies Looking to ..."
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero...","Zoom Children & Family Movies, Comedies Dragge..."


In [10]:
df[(df['type'] == 'Movie') & (df['director'].str.lower().str.contains("nolan"))]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
340,s341,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...","United States, United Kingdom","August 1, 2021",2010,PG-13,148 min,"Action & Adventure, Sci-Fi & Fantasy, Thrillers",A troubled thief who extracts secrets from peo...,"Inception Action & Adventure, Sci-Fi & Fantasy..."


In [11]:
df[(df['listed_in'].str.contains('Action & Adventure, Sci-Fi & Fantasy, Thrillers')) & (df['type'] == 'Movie')]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
340,s341,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...","United States, United Kingdom","August 1, 2021",2010,PG-13,148 min,"Action & Adventure, Sci-Fi & Fantasy, Thrillers",A troubled thief who extracts secrets from peo...,"Inception Action & Adventure, Sci-Fi & Fantasy..."


In [16]:
df[df['country'].str.contains('south korea',case=False)]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
104,s105,TV Show,Tayo the Little Bus,unknown,"Robyn Slade, Kami Desilets",South Korea,"September 6, 2021",2016,TV-Y,2 Seasons,"Kids' TV, Korean TV Shows",As they learn their routes around the busy cit...,"Tayo the Little Bus Kids' TV, Korean TV Shows ..."
124,s125,TV Show,Pororo - The Little Penguin,unknown,unknown,South Korea,"September 2, 2021",2013,TV-Y7,3 Seasons,"Kids' TV, Korean TV Shows","On a tiny island, Pororo the penguin has fun a...","Pororo - The Little Penguin Kids' TV, Korean T..."
193,s194,TV Show,D.P.,unknown,"Jung Hae-in, Koo Kyo-hwan, Kim Sung-kyun, Son ...",", South Korea","August 27, 2021",2021,TV-MA,1 Season,"International TV Shows, TV Dramas",A young private’s assignment to capture army d...,"D.P. International TV Shows, TV Dramas A young..."
289,s290,TV Show,The Crowned Clown,unknown,"Yeo Jin-goo, Lee Se-young, Kim Sang-kyung, Jun...",South Korea,"August 10, 2021",2019,TV-14,1 Season,"International TV Shows, Romantic TV Shows, TV ...","Standing in for an unhinged Joseon king, a loo...","The Crowned Clown International TV Shows, Roma..."
456,s457,TV Show,Her Private Life,unknown,"Park Min-young, Kim Jae-uk, Ahn Bo-hyun, Jung ...",South Korea,"July 15, 2021",2019,TV-14,1 Season,"International TV Shows, Romantic TV Shows, TV ...","An art curator's life unravels, as she tries t...","Her Private Life International TV Shows, Roman..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8484,s8485,Movie,The Rift: The Dark Side of the Moon,Dejan Zečević,"Ken Foree, Katarina Čas, Dragan Mićanović, Mon...","Serbia, South Korea, Slovenia","February 26, 2018",2016,TV-MA,92 min,"Horror Movies, Independent Movies, Sci-Fi & Fa...",When a team of special agents is sent to recov...,The Rift: The Dark Side of the Moon Horror Mov...
8575,s8576,TV Show,This Is My Love,unknown,"Jin-mo Joo, Sa-rang Kim, Junior, Ja-in Lee, Su...",South Korea,"May 22, 2017",2015,TV-14,1 Season,"International TV Shows, Korean TV Shows, Roman...",A renowned actor who is still pining over the ...,"This Is My Love International TV Shows, Korean..."
8613,s8614,Movie,Train to Busan,Sang-ho Yeon,"Gong Yoo, Yu-mi Jung, Dong-seok Ma, Soo-an Kim...",South Korea,"March 18, 2017",2016,TV-MA,118 min,"Action & Adventure, Horror Movies, Internation...","As a zombie outbreak sweeps the country, a dad...","Train to Busan Action & Adventure, Horror Movi..."
8684,s8685,TV Show,Vroomiz,unknown,"Joon-seok Song, Jeong-hwa Yang, Sang-hyun Um, ...",South Korea,"August 1, 2017",2016,TV-Y,3 Seasons,"Kids' TV, Korean TV Shows","For these half-car, half-animal friends, each ...","Vroomiz Kids' TV, Korean TV Shows For these ha..."


In [15]:
df[df['cast'].str.contains("tom cruise",case=False)]

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,overview
341,s342,Movie,Magnolia,Paul Thomas Anderson,"John C. Reilly, Philip Baker Hall, Tom Cruise,...",United States,"August 1, 2021",1999,R,189 min,"Dramas, Independent Movies","Through chance, history and divine interventio...","Magnolia Dramas, Independent Movies Through ch..."
1254,s1255,Movie,Rain Man,Barry Levinson,"Dustin Hoffman, Tom Cruise, Valeria Golino, Ge...",United States,"March 1, 2021",1988,R,134 min,"Classic Movies, Dramas","Motivated by money, a selfish workaholic seeki...","Rain Man Classic Movies, Dramas Motivated by m..."


In [13]:
df.to_csv("/Users/jaimeet/Documents/Movie Recommender/data/netflix_cleaned.csv",index=False)